# dfs-three-set-toposort — ex1: implement three-set DFS topological sort (deps-first, root LAST)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dfs-three-set-toposort`. Running the final beacon cell reports progress against the `Backprop: DFS three-set toposort` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: DFS three-set toposort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dfs-three-set-toposort`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dfs-three-set-toposort"
DD_SUBTOPIC = "Backprop: DFS three-set toposort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DFS three-set toposort — quick refresher

The deps-FIRST topological sort: every node appears AFTER all of its (transitive) children. The root node ends up LAST. This is the lower-level helper that the reverse-pass driver wraps with `[::-1]` to get the end-node-first order.

Classic three-color DFS:

```python
def topological_sort(root, get_children):
    result = []
    perm  = set()   # fully processed (black)
    temp  = set()   # currently on the DFS stack (gray) — cycle detector

    def visit(node):
        nid = id(node)
        if nid in perm: return
        if nid in temp: raise ValueError('cycle')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.remove(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result
```

Returns `[...children..., root]`. Two color-sets, not one: `perm` skips already-finished subtrees in a branching DAG; `temp` catches back-edges (cycles).

### Exercise 1 — implement three-set DFS topological sort (deps-first, root LAST)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the three-color DFS topological-sort algorithm to return descendants of a root in deps-FIRST order (root appears LAST), with a temp-set guard for cycle detection.
> Keywords: dfs, topological-sort, three-color, deps-first
> ```

**KCs targeted:** `dfs-three-set-toposort`, `cycle-detection-temp-set`

Implement `topological_sort(root, get_children)` — the lower-level DAG traversal that batch-4's `sorted_computational_graph` wraps with `[::-1]` for the reverse pass.

**Contract.**
- Returns a `list` of nodes reachable from `root` via `get_children`.
- Every node appears AFTER all of its (transitive) children — deps-first.
- `root` appears LAST. This is the order a FORWARD compute pass would use; reversing it gives the backward order.
- Each reachable node appears EXACTLY once, even in diamond DAGs where multiple paths reach the same descendant.
- Raises `ValueError` on a cycle (we're DAG-only).

**Algorithm.** Classic three-color DFS:
- `temp` (gray) = currently on the recursion stack — cycle detector.
- `perm` (black) = fully processed — skip if already in here.
- (white = not in either set = not yet visited).

```python
def visit(node):
    nid = id(node)
    if nid in perm: return            # already done
    if nid in temp: raise ValueError  # cycle
    temp.add(nid)
    for child in get_children(node):
        visit(child)
    temp.remove(nid)
    perm.add(nid)
    result.append(node)
```

Use `id(node)` as the set key (the test nodes don't override `__hash__`, but it's the safe-by-default identity key).

**This drill is the LOWER half of `sorted_computational_graph`.** Batch-4's atom built the reverse-pass wrapper by composing this helper with `[::-1]`. The cycle-detection logic specifically is the focus of its own sibling atom — feel free to crib that behavior here (the same temp set serves both purposes).

In [ ]:
def topological_sort(root, get_children):
    """DFS topo sort. Returns descendants of root in deps-first order
    (root LAST). Raises ValueError on cycle.
    """
    raise NotImplementedError()


def _test_ex1():
    # --- helper graph node ---
    class N:
        def __init__(self, name, *children):
            self.name = name
            self.children = list(children)
        def __repr__(self):
            return f'N({self.name})'

    def get_children(n):
        return n.children

    # --- linear chain a -> b -> c ---
    c = N('c')
    b = N('b', c)
    a = N('a', b)
    order = topological_sort(a, get_children)
    names = [n.name for n in order]
    assert names[-1] == 'a', f'root must be LAST, got {names}'
    assert names.index('c') < names.index('b') < names.index('a'), names

    # --- diamond DAG ---
    #      a
    #     / \
    #    b   c
    #     \ /
    #      d
    d = N('d')
    b = N('b', d)
    c = N('c', d)
    a = N('a', b, c)
    order = topological_sort(a, get_children)
    names = [n.name for n in order]
    assert names.count('d') == 1, f'd must appear ONCE, got {names}'
    assert names[-1] == 'a', f'root LAST, got {names}'
    assert names.index('d') < names.index('b'), 'd before b (b depends on d)'
    assert names.index('d') < names.index('c'), 'd before c (c depends on d)'
    assert names.index('b') < names.index('a')
    assert names.index('c') < names.index('a')
    assert len(order) == 4, f'four unique nodes, got {len(order)}'

    # --- linked list with shared descendants (long chain) ---
    leaf = N('leaf')
    n3 = N('n3', leaf)
    n2 = N('n2', n3)
    n1 = N('n1', n2)
    n0 = N('n0', n1)
    order = topological_sort(n0, get_children)
    names = [n.name for n in order]
    assert names == ['leaf', 'n3', 'n2', 'n1', 'n0'], (
        f'linear chain must yield deps-first order, got {names}'
    )

    # --- cycle detection raises ValueError ---
    x = N('x')
    y = N('y')
    x.children = [y]
    y.children = [x]
    raised = False
    try:
        topological_sort(x, get_children)
    except ValueError:
        raised = True
    assert raised, 'a cycle must raise ValueError'

    # --- self-loop ---
    s = N('s')
    s.children = [s]
    raised = False
    try:
        topological_sort(s, get_children)
    except ValueError:
        raised = True
    assert raised, 'self-loop must raise ValueError'

    # --- singleton (root with no children) ---
    lonely = N('lonely')
    order = topological_sort(lonely, get_children)
    assert order == [lonely], f'singleton: {order}'

    # --- mid-graph cycle does NOT silently succeed ---
    # Graph: a -> b -> c -> b (cycle through b).
    p = N('p')
    q = N('q')
    r = N('r')
    p.children = [q]
    q.children = [r]
    r.children = [q]  # cycle
    raised = False
    try:
        topological_sort(p, get_children)
    except ValueError:
        raised = True
    assert raised, 'mid-graph cycle must raise ValueError'

    # --- branching where one branch is deep ---
    # root -> a -> b -> c
    #      -> d
    cc = N('c')
    bb = N('b', cc)
    aa = N('a', bb)
    dd = N('d')
    root = N('root', aa, dd)
    order = topological_sort(root, get_children)
    names = [n.name for n in order]
    # Just check the deps-first invariant, not a specific ordering.
    pos = {nm: i for i, nm in enumerate(names)}
    assert pos['c'] < pos['b'] < pos['a'] < pos['root']
    assert pos['d'] < pos['root']
    assert names[-1] == 'root'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def topological_sort(root, get_children):
    result = []
    perm = set()   # fully processed (black) — keyed by id()
    temp = set()   # currently on DFS stack (gray) — cycle detector

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError(f'Cycle detected at {node!r} — graph is not a DAG')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.remove(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result
```

**Two color sets — `temp` and `perm` — do different jobs.** `perm` is the 'I have finished this subtree, do not recurse again' marker — it keeps a diamond DAG from re-traversing the shared descendant. `temp` is the 'I am currently inside this subtree' marker — re-entering it means a back-edge → cycle. Two sets, two semantics.

**Why `id(node)` instead of the node itself.** The graph nodes in the tests are simple Python objects with identity-equality, so a plain `set` of nodes would work. But the same code runs on MiniTensors (where `__eq__` might compare by value if you ever add it), or numpy arrays (where `__eq__` returns an array). `id()` is the safe-by-default identity key.

**Output order is deps-FIRST.** The result list ends with the root, which is what a forward compute pass expects: evaluate leaves first, then their consumers. The reverse pass needs the OPPOSITE — and that's exactly the `[::-1]` you'll see in the sibling `sorted-computational-graph` atom. Keeping this helper deps-first means it's reusable for forward-only graph operations (`zero_grad`, structural-check passes).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()